In [1]:
# 06_utility_sensitivity.ipynb -- Sensitivity of the utility proxy to its hand-set weights, with bootstrap CIs over files.
# Addresses the reviewer concern that the C2>C3 ordering and dominance verdict depend on undisclosed weights.
# Reads masked_corpus.json (per-file identifier/literal masking counts) produced by notebook 02. No LLM calls.
# Output paths are intentionally not printed (anonymity).

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys"})

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA, FIG, TAB = ROOT / "data", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)

with open(DATA / "masked_corpus.json", encoding="utf-8") as f:
    masked = json.load(f)
TIERS = ["C0", "C1", "C2", "C3", "C4"]

# Per-file masked fractions of identifiers and literals, relative to the totals exposed by C2/C3.
tot_id = sum(r["identifier"] for r in masked["C2"]) or 1
tot_lit = sum(r["literal"] for r in masked["C3"]) or 1

def per_file_fracs(tier):
    # Return arrays (over files) of identifier-masked and literal-masked fractions at this tier.
    id_by = {r["file_id"]: r["identifier"] for r in masked[tier]}
    lit_by = {r["file_id"]: r["literal"] for r in masked[tier]}
    id_tot = {r["file_id"]: r["identifier"] for r in masked["C2"]}
    lit_tot = {r["file_id"]: r["literal"] for r in masked["C3"]}
    fids = sorted(id_by)
    idm = np.array([id_by[f] / id_tot[f] if id_tot[f] else 0.0 for f in fids])
    litm = np.array([lit_by[f] / lit_tot[f] if lit_tot[f] else 0.0 for f in fids])
    return idm, litm

def utility(tier, w_floor, w_id, w_lit):
    idm, litm = per_file_fracs(tier)
    return w_floor + w_id * (1 - idm) + w_lit * (1 - litm)  # per-file utility array

# --- Baseline weights (now disclosed in the manuscript) --------------------
W_FLOOR, W_ID, W_LIT = 0.30, 0.45, 0.25

# --- Bootstrap CIs on utility per tier at baseline weights ------------------
B = 10000
rows = []
for tier in TIERS:
    u = utility(tier, W_FLOOR, W_ID, W_LIT)
    idx = rng.integers(0, len(u), size=(B, len(u)))
    bmeans = u[idx].mean(axis=1)
    rows.append({"tier": tier, "utility_mean": float(u.mean()),
                 "ci_lo": float(np.percentile(bmeans, 2.5)),
                 "ci_hi": float(np.percentile(bmeans, 97.5))})
util_ci = pd.DataFrame(rows)
util_ci.to_csv(TAB / "utility_ci.csv", index=False)

# --- Weight-simplex sweep: does mean(C2 utility) > mean(C3 utility) hold? ----
# Sweep (w_id, w_lit) with w_floor = 1 - w_id - w_lit >= 0.1 (keep a minimum skeleton floor).
grid = np.linspace(0.0, 0.9, 46)
flip = []
c2c3_gap = np.full((len(grid), len(grid)), np.nan)
for i, wid in enumerate(grid):
    for j, wlit in enumerate(grid):
        wfloor = 1 - wid - wlit
        if wfloor < 0.1:
            continue
        c2 = utility("C2", wfloor, wid, wlit).mean()
        c3 = utility("C3", wfloor, wid, wlit).mean()
        c2c3_gap[i, j] = c2 - c3
        flip.append(c2 > c3)
frac_c2_gt_c3 = float(np.mean(flip)) if flip else float("nan")
pd.DataFrame({"frac_weightings_with_C2_gt_C3": [frac_c2_gt_c3]}).to_csv(TAB / "utility_weight_robustness.csv", index=False)

# --- Figure: utility by tier with bootstrap CIs ----------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.4))
y = util_ci["utility_mean"].to_numpy()
lo = util_ci["ci_lo"].to_numpy(); hi = util_ci["ci_hi"].to_numpy()
ax.errorbar(TIERS, y, yerr=np.vstack([y - lo, hi - y]), color="0.15", marker="o",
            capsize=3, linewidth=1.6)
ax.set_xlabel("Masking tier")
ax.set_ylabel("Utility proxy (95% CI)")
ax.set_ylim(-0.03, 1.03)
save_fig(fig, "fig_utility_ci")

# --- Figure: heatmap of C2-C3 utility gap over the weight simplex ----------
fig, ax = plt.subplots(figsize=(4.8, 4.0))
im = ax.imshow(c2c3_gap, origin="lower", extent=[grid[0], grid[-1], grid[0], grid[-1]],
               cmap="Greys", aspect="auto", vmin=np.nanmin(c2c3_gap), vmax=np.nanmax(c2c3_gap))
cs = ax.contour(grid, grid, c2c3_gap, levels=[0.0], colors="black", linewidths=1.5)
ax.clabel(cs, fmt="C2=C3", fontsize=8)
ax.scatter([W_ID], [W_LIT], s=60, facecolors="white", edgecolors="black", linewidths=1.3, zorder=3)
ax.annotate("baseline", (W_ID, W_LIT), textcoords="offset points", xytext=(6, 6), fontsize=9)
ax.set_xlabel("Identifier weight")
ax.set_ylabel("Literal weight")
fig.colorbar(im, ax=ax, label="C2 - C3 utility")
save_fig(fig, "fig_utility_weight_heatmap")

print("Utility by tier (baseline weights, 95% CI):")
print(util_ci.to_string(index=False))
print(f"\nFraction of valid weightings with mean utility C2 > C3: {frac_c2_gt_c3:.3f}")


Utility by tier (baseline weights, 95% CI):
tier  utility_mean  ci_lo  ci_hi
  C0          1.00   1.00   1.00
  C1          1.00   1.00   1.00
  C2          0.55   0.55   0.55
  C3          0.30   0.30   0.30
  C4          0.30   0.30   0.30

Fraction of valid weightings with mean utility C2 > C3: 0.957
